# RAG Chat Application (Notebook Version)

This notebook implements a Retrieval Augmented Generation (RAG) pipeline:
1. Load a PDF document
2. Split it into chunks and store in FAISS vector database
3. Ask questions — relevant chunks are retrieved and sent to Ollama LLM
4. All LLM calls are traced via Langfuse

**Prerequisites:** Ollama running with `llama3.2:3b` model pulled

## 1. Install Dependencies

In [57]:
!pip install langchain langchain-community langchain-text-splitters langchain-ollama langfuse faiss-cpu pypdf python-dotenv -q

## 2. Imports & Environment Setup

In [58]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langfuse.langchain import CallbackHandler

load_dotenv()
print("Environment loaded successfully!")

Environment loaded successfully!


## 3. Load and Process the PDF

In [59]:
# Load PDF — change this path to your PDF file
PDF_PATH = "Gopalswamy Doraiswamy Naidu - Wikipedia.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

print(f"Loaded {len(documents)} pages from the PDF")

Loaded 4 pages from the PDF


In [60]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print(f"\nSample chunk (first one):\n{chunks[0].page_content[:300]}...")

Split into 13 chunks

Sample chunk (first one):
Gopalaswamy Doraiswamy Naidu
Gopalaswamy Doraiswamy Naidu
Born 23 March 1893
Kalangal, Coimbatore District,
Madras Presidency, British India
(now Tamil Nadu, India)
Died 4 January 1974 (aged 80)
Coimbatore, Tamil Nadu, India
CitizenshipIndian
Known for Technical redesigner, Industrialist,
businessma...


## 4. Create Vector Store (FAISS)

In [61]:
# Load embedding model and build FAISS index
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

print("FAISS vector store created with", len(chunks), "vectors")

FAISS vector store created with 13 vectors


## 5. Test Retrieval
Let's verify the retriever finds relevant chunks before hooking up the LLM.

In [62]:
# Test retrieval with a sample query
test_query = "when was G.D. Naidu born?"
retrieved_docs = retriever.invoke(test_query)

for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get("page", "?")
    print(f"\n--- Chunk {i} (Page {page}) ---")
    print(doc.page_content[:300], "...")


--- Chunk 1 (Page 2) ---
produced in which R. Madhavan plays the role of Naidu. [15]
1. "Coimbatore's wealth creators" (https://web.archive.org/web/20040510163921/http://www.hind
u.com/mp/2004/02/02/stories/2004020201050100.htm). The Hindu. Coimbatore, India. 2
February 2004. Archived from the original (http://www.hindu.com ...

--- Chunk 2 (Page 2) ---
understatement!" He is survived by his son G.D. Gopal and grandchildren G.D. Rajkumar and
Shantini. A permanent Industrial Exhibition in his memory is in Coimbatore. He provided
employment in the engineering and manufacturing sectors to many individuals in the 1950s and
1960s.
G. D. Matriculation Hi ...

--- Chunk 3 (Page 2) ---
and-culture/article1988214.ece). The Hindu. India. 3 May 2011.
10. "GD Naidu | PDF" (https://www.scribd.com/presentation/135297775/Gd-Naidu). Scribd.
11. https://www.scribd.com/doc/135297775/Gd-Naidu/ – Gd Naidu Scribd
12. "G.D Matriculation Higher Secondary School" (https://www.gdschoolcbe.com/abou ...


## 6. Initialize LLM (Ollama)

In [63]:
llm = ChatOllama(model="llama3.2:3b", temperature=0.3)

# Quick test to make sure Ollama is running
test_response = llm.invoke([HumanMessage(content="Say hello in one word.")])
print("Ollama is working! Response:", test_response.content)

Ollama is working! Response: Hello.


## 7. RAG Query Function

In [64]:
# Chat history for multi-turn conversation
chat_history = []


def ask(question: str) -> str:
    """Ask a question about the PDF. Maintains conversation history."""
    # Retrieve relevant chunks
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Build messages
    messages = [
        SystemMessage(content=(
            "You are a helpful assistant that answers questions based on the provided context from a PDF document. "
            "Use ONLY the context below to answer. If the answer is not in the context, say so.\n\n"
            f"Context:\n{context}"
        )),
    ]
    messages.extend(chat_history)
    messages.append(HumanMessage(content=question))

    # Call LLM with Langfuse tracing
    langfuse_handler = CallbackHandler()
    response = llm.invoke(messages, config={"callbacks": [langfuse_handler]})
    answer = response.content

    # Update chat history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    # Print sources
    print("\n" + "=" * 60)
    print(f"Question: {question}")
    print("=" * 60)
    print(f"\nAnswer:\n{answer}")
    print(f"\n--- Sources ({len(docs)} chunks) ---")
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get("page", "?")
        print(f"  Chunk {i} | Page {page} | {doc.page_content[:100]}...")

    return answer


print("RAG function ready! Use ask('your question') to chat with the PDF.")

RAG function ready! Use ask('your question') to chat with the PDF.


## 8. Ask Questions!
Run the cells below or add your own `ask(...)` calls.

In [65]:
ask("When was G.D. Naidu born?")


Question: When was G.D. Naidu born?

Answer:
G.D. Naidu was born on 23 March 1893.

--- Sources (3 chunks) ---
  Chunk 1 | Page 2 | produced in which R. Madhavan plays the role of Naidu. [15]
1. "Coimbatore's wealth creators" (https...
  Chunk 2 | Page 0 | were primarily industrial but also spanned the fields
of electrical, mechanical, agricultural (hybri...
  Chunk 3 | Page 2 | understatement!" He is survived by his son G.D. Gopal and grandchildren G.D. Rajkumar and
Shantini. ...


'G.D. Naidu was born on 23 March 1893.'

In [66]:
ask("When did he pass away?")


Question: When did he pass away?

Answer:
G.D. Naidu passed away on 4 January 1974.

--- Sources (3 chunks) ---
  Chunk 1 | Page 2 | understatement!" He is survived by his son G.D. Gopal and grandchildren G.D. Rajkumar and
Shantini. ...
  Chunk 2 | Page 1 | researched and identified new varieties in cotton, maize
and papaya. His farm was visited by Sir C. ...
  Chunk 3 | Page 1 | college.[10] Naidu was not satisfied with the four-year programs and said that it was a waste of tim...


'G.D. Naidu passed away on 4 January 1974.'

In [67]:
ask("What was my first question?")


Question: What was my first question?

Answer:
Your first question was "When was G.D. Naidu born?"

--- Sources (3 chunks) ---
  Chunk 1 | Page 2 | produced in which R. Madhavan plays the role of Naidu. [15]
1. "Coimbatore's wealth creators" (https...
  Chunk 2 | Page 0 | business in 1920, with the purchase of an automobile coach. He drove it between Pollachi and
Palani....
  Chunk 3 | Page 1 | measures including grants for research scholarships and
welfare schemes for his employees and the de...


'Your first question was "When was G.D. Naidu born?"'

## 9. Interactive Chat Loop (Optional)
Run this cell for a continuous chat experience. Type `quit` to stop.

In [68]:
# Reset history for a fresh conversation
chat_history.clear()
print("Chat started! Type 'quit' to exit.\n")

while False:
    question = input("You: ")
    if question.strip().lower() in ("quit", "exit", "q"):
        print("Chat ended.")
        break
    if not question.strip():
        continue
    ask(question)

Chat started! Type 'quit' to exit.



In [69]:
from langchain_community.chat_message_histories import ChatMessageHistory

memory = ChatMessageHistory()


def ask_with_memory(question: str) -> str:
    """Ask a question about the PDF using LangChain memory."""
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    messages = [
        SystemMessage(content=(
            "You are a helpful assistant that answers questions based on the provided context from a PDF document. "
            "Use ONLY the context below to answer. If the answer is not in the context, say so.\n\n"
            f"Context:\n{context}"
        )),
        *memory.messages,
        HumanMessage(content=question),
    ]

    langfuse_handler = CallbackHandler()
    response = llm.invoke(messages, config={"callbacks": [langfuse_handler]})
    answer = response.content

    memory.add_user_message(question)
    memory.add_ai_message(answer)

    print(f"\n{'='*60}\nQuestion: {question}\n{'='*60}\n\nAnswer:\n{answer}")
    print(f"\n--- Sources ({len(docs)} chunks) ---")
    for i, doc in enumerate(docs, 1):
        print(f"  Chunk {i} | Page {doc.metadata.get('page', '?')} | {doc.page_content[:100]}...")

    return answer


print("RAG function ready! Use ask_with_memory('your question') to chat with the PDF.")

RAG function ready! Use ask_with_memory('your question') to chat with the PDF.


In [70]:
ask_with_memory("When was G.D. Naidu born?")


Question: When was G.D. Naidu born?

Answer:
G.D. Naidu was born on 23 March 1893.

--- Sources (3 chunks) ---
  Chunk 1 | Page 2 | produced in which R. Madhavan plays the role of Naidu. [15]
1. "Coimbatore's wealth creators" (https...
  Chunk 2 | Page 0 | were primarily industrial but also spanned the fields
of electrical, mechanical, agricultural (hybri...
  Chunk 3 | Page 2 | understatement!" He is survived by his son G.D. Gopal and grandchildren G.D. Rajkumar and
Shantini. ...


'G.D. Naidu was born on 23 March 1893.'